# 7주차 실습 — SQL (1) 조회 기초

**오늘 끝나면 할 수 있어야 하는 것**

1. SQL 문장의 종류를 구분할 수 있다
2. SELECT · WHERE로 데이터를 조회한다
3. 오류 메시지를 읽고 고칠 수 있다

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` 같은 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.

In [ ]:
#@title [1번] ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

ANSWERS = _load_answers("eNqdlV9P01AYxr/KyW6QRA3rv60mXqjsgggkRhJjjDHt2inJZGZgvDAmHRadbIkMKCvJNodOBjpjkYEYiR/Gy76n38H3lE1PJybMXWzr07Oz/p73ed/zNHYjHrtEnsbmZ+/jZ0wR9WRCSWuSKJux8ySW6+myIEqJZFxMpg2B6fnck3nUxYQ8puJlOpdll3disLYHW3X4UgzcI7aOtu1g4xV9Xgie12J3UXgwO7fANryZmkxdmyGw2qLVFQL7x4RWu4TWu7Bt07pNoHMUOJ/YPdosBI0yocUatWtQtmCrxjS6fAClIpRaFwl82IG1JqGlWlB1cQsCiy7dtKhzTALHhc/hTyYnpiZm8A8sAoebuIi2LUKd5T/bhLh5w8ybBj5gRsvOm0x5tDCbm9OyfekZajeEqGWaFFc0RdGMBG/ZoN6zLM7b1XvkohvxBipIveX5e93QCaRDAdplurUXON0+1m6XXJkeJ7Sx1Pfhp8Xo0eyf1iYD9b8eEd9bhdc2CV53AmcncHb7a2nxI22swPI6gXWPut2ggq5WXzD36HpheEtE3hJRzeiqaCQFQ1Z4Swb1s1tyNTVzK5WaDutHNyoECnXmzV9YsNik7RrDGrmNrwtTUxfGx0dI4DoYwDA5jQruwTE7xV7I/isJUiQJsmDoaV1W0kqkeQb1s2NPTJNzWEFcie/CKGFgtNqBNx0mENqqhFFYdKF+HNay9p3LA6t9m31lNT4Fnkm+t0LLr4YHl3lw00youqGYqi5nePBB/ezgkxPXUyEtNjx9UcZpwNL6/pj4Bx04tPmKDy7wLNju9Lrb38ex4P0IN/J24YvDQvM73+QcLbXwn4k0Ogy6wqNLuqaZQlLNCJLAow/qp6FDacfft8OuK0cHwDebbmAjNor45IjCnv8yYeXCO/7BCtZ8cHz9ZomPXiQj01dGQvdwIuDEgNK3k94uh4Oh2wuLf2ShZywlJxpuXmRXPg5j3BG2l4aPRYL3Rtbigi4m1aRujPHeqEZc1QVF0zNmmvNG5r2ZSy/cmzXYXXMun8tmH5poDe9RUPDCgB9age1h/zrQ7LBeaLbYZBxP3bx2Hg1qYHPjsYFD9CV2i41DFM+abm9R70honM65kH/8T8wkj2nIibiu6II0JkTa3sTpL5iiJEqmwWFGzstHD7R5M8KFPQ37B2FOsWROk0U4LJFL6NsleO9GGfrnqLcexuPkOA0c/FrawQyxjgD73dCz7dkvBmeRqQ==")   # 자동 확인용 (정답 SQL 은 없음)

## 1. SQL 종류와 실습 환경 — DDL · DML · DCL

**SQL의 세 가지 종류**

| 종류 | 하는 일 | 대표 명령 | 예시 |
|---|---|---|---|
| DDL 정의어 | 표의 구조를 만들고 바꾼다 | `CREATE` · `ALTER` · `DROP` | 환자 표를 만든다 |
| DML 조작어 | 데이터를 꺼내고 바꾼다 | `SELECT` · `INSERT` · `UPDATE` · `DELETE` | 당뇨 환자 목록을 뽑는다 |
| DCL 제어어 | 사용 권한을 주고 거둔다 | `GRANT` · `REVOKE` | 연구원에게 권한을 준다 |

오늘은 DML 중 SELECT(조회)를 주로 배웁니다.

DDL · DCL은 아래에서 문장을 읽을 수 있을 만큼만 짚고 넘어갑니다.

아래 SQL은 **읽기만** 합니다.

### DDL - 표 만들기
“환자 표를 만들어 줘”

```sql
CREATE TABLE 환자 (
   환자번호  INT          PRIMARY KEY,
   이름      VARCHAR(20)  NOT NULL,
   생년월일  DATE,
   성별      CHAR(1)      CHECK (성별 IN ('M','F')),
   진료과    VARCHAR(10)  REFERENCES 진료과(코드)
);
```

| 규칙 | 뜻 |
|---|---|
| `PRIMARY KEY` | 기본키. 겹치거나 비면 안 됨 |
| `NOT NULL` | 빈 값(NULL) 금지 |
| `CHECK` | 정해진 값만 허용 |
| `REFERENCES` | 외래키. 다른 표에 있는 값만 |

INT · VARCHAR · DATE는 5주차에 배운 데이터 타입입니다.

실습 표는 미리 만들어 두었습니다.

성별 칸의 `IN ('M','F')` 는 M과 F만 허용한다는 뜻이고, 진료과 표는 이미 있다고 가정합니다.

“병원 표 세 개를 만들어 줘”

```sql
CREATE TABLE 의사     (의사번호 INT PRIMARY KEY, 이름 VARCHAR(20) NOT NULL, 진료과 VARCHAR(10));
CREATE TABLE 환자     (환자번호 INT PRIMARY KEY, 이름 VARCHAR(20) NOT NULL, 생년월일 DATE);
CREATE TABLE 진료기록 (진료번호 INT PRIMARY KEY,
                       의사번호 INT REFERENCES 의사(의사번호),
                       환자번호 INT REFERENCES 환자(환자번호));
```

| 표 | 담는 내용 |
|---|---|
| 의사 | 의사번호 · 이름 · 진료과 |
| 환자 | 환자번호 · 이름 · 생년월일 |
| 진료기록 | 누가 누구를 언제 진료했는지 |

진료기록 표는 `REFERENCES` 로 의사 표와 환자 표를 연결합니다 (외래키).

### DDL - 표 고치기와 지우기

| 하고 싶은 일 | SQL | 결과 |
|---|---|---|
| 열 추가 | `ALTER TABLE 환자 ADD 키 INT;` | 키 열이 새로 생긴다 |
| 열 크기 변경 | `ALTER TABLE 환자 ALTER 이름 TYPE VARCHAR(30);` | 이름을 30자까지 담는다 |
| 열 이름 변경 | `ALTER TABLE 환자 RENAME COLUMN 환자번호 TO ID;` | 열 이름만 바뀐다 |
| 열 삭제 | `ALTER TABLE 환자 DROP COLUMN 생년월일;` | 열과 그 안의 값이 사라진다 |
| 표 삭제 | `DROP TABLE 환자;` | 표 전체가 사라진다 |

DROP은 데이터까지 지우고 되돌릴 수 없어 관리자만 씁니다.

ALTER 문법은 DBMS마다 조금씩 다르며, 여기는 DuckDB · PostgreSQL 기준입니다.

### DML - 데이터 꺼내고 바꾸기

| 하고 싶은 일 | SQL | 결과 |
|---|---|---|
| 조회 | `SELECT 이름 FROM 환자 WHERE 진료과 = '내과';` | 내과 환자 이름이 나온다 |
| 추가 | `INSERT INTO 환자 VALUES (1001, '김하늘', …);` | 새 환자 한 명이 생긴다 |
| 수정 | `UPDATE 환자 SET 진료과 = '외과' WHERE 환자번호 = 1001;` | 진료과가 외과로 바뀐다 |
| 삭제 | `DELETE FROM 환자 WHERE 환자번호 = 1001;` | 그 환자의 행이 사라진다 |

SELECT는 오늘부터 배우고, INSERT · UPDATE · DELETE는 10주차에 실습합니다.

### DCL · TCL

DCL은 누가 어떤 표를 볼지 정하고, TCL은 변경을 확정하거나 되돌립니다.

| 구분 | 명령 | 하는 일 | 예시 |
|---|---|---|---|
| DCL 권한 | `GRANT` | 권한을 준다 | `GRANT SELECT ON 환자 TO 연구원;` |
| | `REVOKE` | 준 권한을 거둔다 | `REVOKE SELECT ON 환자 FROM 연구원;` |
| TCL 트랜잭션 | `COMMIT` | 변경을 확정한다 | `COMMIT;` |
| | `ROLLBACK` | 확정 전 변경을 되돌린다 | `ROLLBACK;` |
| | `SAVEPOINT` | 되돌아갈 지점을 표시한다 | `SAVEPOINT 저장1;` |

**실습 환경** — 설치 없이 웹 브라우저에서 SQL을 씁니다.

① 노트북 열기 (Colab 링크 클릭) → ② 준비 셀 실행 (데이터 자동 연결) → ③ SQL 작성 (`%%sql` 셀에 입력) → ④ 결과 확인 (표로 바로 출력)

- **DuckDB** — 파일 하나로 된 관계형 DBMS. 문법은 PostgreSQL과 거의 같습니다.
- **jupysql** — 노트북에서 SQL을 쓰는 도구. `%%sql` 로 시작하는 칸에 SQL만 입력합니다.

데이터는 읽기 전용으로 엽니다. 실수로 지우거나 바꿀 수 없습니다.

데이터를 바꾸는 명령은 10주차에서 다룹니다.

**실습 데이터 — aact_mini** — 파일 하나에 표 14개가 들어 있는 수업용 데이터베이스

| aact_mini (DB 파일 1개) | |
|---|---|
| 원본 | AACT (ClinicalTrials.gov) |
| 추출일 | 2026-09-19 |
| 범위 | 2015~2024년 시작 시험 |
| 시험 수 | 37,509건 |
| 표 | 14개, 모두 nct_id로 연결 |

- **오늘 쓰는 표**: `studies`
- **주요 표**: `eligibilities` · `sponsors` · `conditions` · `browse_conditions` · `countries` · `facilities`
- **그 밖의 표**: `brief_summaries` · `interventions` · `browse_interventions` · `calculated_values` · `designs` · `design_outcomes` · `keywords`

**aact_mini 데이터 구성 (ERD)**

![aact_mini 데이터 구성 (ERD)](https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini_erd.png)

아래 셀을 실행해 표 이름 14개가 나오면 연결된 것입니다.

In [ ]:
%%sql
-- [2번] 어떤 표(테이블)가 있는지
SELECT table_name
FROM information_schema.tables;

## 2. SELECT · FROM — 필요한 열 고르기

**SQL 문장의 구조** — 아래 다섯 부분을 순서대로 적습니다.

| 절 | 뜻 | 예 |
|---|---|---|
| `SELECT` | 무엇을 볼지 | `nct_id, brief_title` |
| `FROM` | 어디에서 (테이블) | `studies` |
| `WHERE` | 어떤 행만 (조건) | `phase = 'PHASE3'` |
| `ORDER BY` | 어떤 순서로 (정렬) | `start_date DESC` |
| `LIMIT` | 몇 줄만 (개수) | `10` |

순서를 지키지 않으면 오류가 납니다.

**적는 순서와 처리 순서**

| | 순서 |
|---|---|
| 적는 순서 (사람의 순서) | SELECT › FROM › WHERE › ORDER BY › LIMIT |
| 처리 순서 (컴퓨터의 순서) | FROM › WHERE › SELECT › ORDER BY › LIMIT |

**왜 SELECT를 먼저 적나** — SQL의 원래 이름은 SEQUEL(Structured English Query Language)입니다. 영어 문장처럼 읽힙니다.

`Show titles from studies where phase is 3` → `SELECT brief_title FROM studies WHERE …`

**왜 SELECT를 나중에 처리하나** — 남을 행이 정해져야 그 행의 열을 고를 수 있습니다.

1. `FROM` — 표를 가져와 행을 만듦
2. `WHERE` — 남길 행을 정함
3. `SELECT` — 남은 행마다 보여 줄 열을 만듦

그래서 SELECT에 없는 `phase`로도 거를 수 있습니다.

In [ ]:
%%sql
-- [3번] 예시 쿼리 — phase는 결과에 없지만 이미 거르는 데 쓰였음
SELECT brief_title
FROM studies
WHERE phase = 'PHASE3';

**SELECT · FROM**

`LIMIT 3` 은 앞에서 3줄만 보라는 뜻입니다. 큰 표를 처음 볼 때 꼭 붙입니다.

`ORDER BY` 없이 쓰면 어떤 3줄이 나올지 정해져 있지 않아 사람마다 결과가 다를 수 있습니다.

In [ ]:
%%sql
-- [4번] “임상시험의 등록번호와 제목을 보여 줘”
SELECT nct_id, brief_title
FROM studies
LIMIT 3;

**SELECT \* - 전체 열 보기**

In [ ]:
%%sql
-- [5번] “studies 표에 어떤 열들이 있는지 보여 줘”
SELECT *
FROM studies
LIMIT 3;

**AS - 결과 열 이름 바꾸기** — 보고서에 그대로 쓸 수 있게 이름을 붙입니다.

`AS` 는 생략할 수 있지만, 적어 두면 나중에 읽기 훨씬 좋습니다.

띄어쓰기가 있는 이름은 `"등록 인원"`처럼 큰따옴표를 사용합니다.

In [ ]:
%%sql
-- [6번] “등록번호와 등록인원을 한글 이름으로 보여 줘”
SELECT nct_id AS 등록번호,
       enrollment AS 등록인원
FROM studies
LIMIT 3;

**COUNT(\*) - 행 개수 세기** — `COUNT(*)`는 행이 몇 개인지 셉니다.

In [ ]:
%%sql
-- [7번] “이 표에 행이 몇 개나 있어?”  → 37,509
SELECT COUNT(*) AS 시험수
FROM studies;

**AS 없이 실행하면** — 숫자는 같고, 열 이름만 DuckDB가 자동으로 붙인 이름입니다.

In [ ]:
%%sql
-- [8번] → 열 이름이 count_star()
SELECT COUNT(*)
FROM studies;

## 3. WHERE — 필요한 행 고르기

**WHERE 조건에 쓰는 기호** — 조건은 이 여섯 가지 모양이면 거의 다 됩니다.

| 종류 | 쓰는 법 | 예 | 뜻 |
|---|---|---|---|
| 비교 | `=` `<>` `>` `>=` `<` `<=` (`<>` 는 ‘같지 않다’) | `enrollment >= 1000` | 등록인원 1000명 이상 |
| 여러 조건 | `AND` `OR` `NOT` | `A AND B` | 두 조건을 모두 만족 |
| 목록 | `IN (값1, 값2)` | `phase IN ('PHASE3','PHASE4')` | 여러 값 중 하나 |
| 범위 | `BETWEEN a AND b` | `start_date BETWEEN '2024-01-01' AND '2024-12-31'` | 양 끝을 포함한 범위 |
| 글자 | `LIKE '%글자%'` | `brief_title LIKE '%Nurs%'` | 그 글자가 들어간 것 |
| 빈 값 | `IS NULL` / `IS NOT NULL` | `results_first_posted_date IS NULL` | 값이 없는 / 있는 행 |

**WHERE - 조건 하나** — 완료된 시험만 보기

문자 값 `'COMPLETED'` 는 작은따옴표를 사용합니다.

큰따옴표는 테이블명, 컬럼명, 별칭(Alias) 같은 데이터베이스 객체의 이름 등에만 사용합니다.

In [ ]:
%%sql
-- [9번] “완료(COMPLETED)된 시험이 몇 건이야?”  → 19,622
SELECT COUNT(*) AS 완료시험
FROM studies
WHERE overall_status = 'COMPLETED';

**WHERE - 조건 두 개 (AND)** — 조건을 하나씩 더해 갑니다.

조건을 하나씩 추가하면서 건수가 어떻게 줄어드는지 확인해야 합니다.

In [ ]:
%%sql
-- [10번] “2024년에 시작한 3상 시험을 보여 줘”
SELECT nct_id, start_date, phase
FROM studies
WHERE start_date BETWEEN '2024-01-01'
                    AND '2024-12-31'
  AND phase = 'PHASE3'
ORDER BY start_date DESC
LIMIT 3;

**LIKE - 글자가 들어간 행 찾기** — IN(목록)과 BETWEEN(범위)은 위의 기호 표를 참고하세요.

In [ ]:
%%sql
-- [11번] “제목에 Nurs 가 들어간 시험은 몇 건이야?”  → 187
SELECT COUNT(*) AS 간호관련
FROM studies
WHERE brief_title LIKE '%Nurs%';

LIKE의 두 기호: `%` = 아무 글자 0개 이상, `_` = 정확히 한 글자

| 패턴 | 뜻 | 찾음 | 못 찾음 |
|---|---|---|---|
| `LIKE 'Nurs%'` | Nurs로 시작 | Nursing Care | Home Nurse Visit |
| `LIKE '%Nurs'` | Nurs로 끝남 | Pediatric Nurs | Nursing Care |
| `LIKE '%Nurs%'` | 어디든 Nurs가 들어감 | Home Nurse Visit | nursing (소문자) |
| `LIKE 'PHASE_'` | PHASE 뒤에 딱 한 글자 | PHASE1, PHASE3 | PHASE1/PHASE2 |

**IS NULL - 값이 없는 행 찾기**

In [ ]:
%%sql
-- [12번] “완료됐는데 결과가 게시되지 않은 시험은?”  → 16,098
SELECT COUNT(*) AS 결과없음
FROM studies
WHERE overall_status = 'COMPLETED'
  AND results_first_posted_date IS NULL;

`= NULL` 로 쓰면 오류 없이 0이 나오니 주의하세요.

‘NA’처럼 ‘없음’을 글자로 적은 값은 `IS NULL`로 잡히지 않습니다 (8주차에서 다룸).

**IS NOT NULL - 값이 있는 행 찾기**

In [ ]:
%%sql
-- [13번] “완료됐고 결과도 게시된 시험은?”  → 3,524
SELECT COUNT(*) AS 결과있음
FROM studies
WHERE overall_status = 'COMPLETED'
  AND results_first_posted_date IS NOT NULL;

앞의 16,098건과 더하면 완료 시험 19,622건이 됩니다.

`IS NULL`과 `IS NOT NULL`은 빠짐없이 둘로 나눕니다.

`<> NULL` 로 써도 0이 나옵니다.

## 4. 정렬과 중복 제거 — ORDER BY · DISTINCT

**ORDER BY - 정렬** — 큰 값부터, 또는 최근 순으로

In [ ]:
%%sql
-- [14번] “등록인원이 가장 많은 시험 3건을 보여 줘”
SELECT nct_id, enrollment
FROM studies
ORDER BY enrollment DESC
LIMIT 3;

1억 8천만 명은 실제 등록 인원일 수 없습니다.

정렬은 이런 이상한 값을 찾는 데도 씁니다.

값이 없는(NULL) 행은 DuckDB에서 맨 뒤에 놓입니다.

**DISTINCT - 값의 종류 보기** — 이 열에 어떤 값들이 있는지 먼저 확인합니다.

In [ ]:
%%sql
-- [15번] “진행 상태에 어떤 값들이 있어?”  → 모두 9가지
SELECT DISTINCT overall_status
FROM studies
ORDER BY overall_status;

WHERE 조건을 쓰기 전에 `DISTINCT` 로 값의 종류를 먼저 보면 오타와 빈 결과를 막을 수 있습니다.

열을 두 개 적으면 두 값의 조합으로 중복을 없애고, NULL도 한 가지 값으로 나옵니다.

## 5. 자주 하는 실수 — 오류 메시지 읽기

`[오류 예시]` 셀은 일부러 틀리게 적었습니다.

### 실수 1: `= NULL`

오류가 나지 않고 결과가 0행으로 나옵니다. NULL은 ‘같다/다르다’로 비교 자체를 할 수 없기 때문입니다.

In [ ]:
%%sql
-- [16번] [오류 예시] 이렇게 쓰면 → 0행
SELECT nct_id
FROM studies
WHERE results_first_posted_date = NULL;

In [ ]:
%%sql
-- [17번] 이렇게 고칩니다 → IS NULL
SELECT nct_id
FROM studies
WHERE results_first_posted_date IS NULL;

결과가 0행이면 “없다”가 아니라 **“조건이 틀렸다”** 를 먼저 의심하세요.

**왜 0행일까** — NULL과 비교하면 ‘알 수 없음’이 되고, WHERE는 참인 행만 남깁니다.

| 비교 | 결과 | WHERE에서 | 이유 |
|---|---|---|---|
| `'PHASE3' = 'PHASE3'` | 참 | 남김 | 같은 값 |
| `'PHASE3' = NULL` | 알 수 없음 | 버림 | 모르는 값이 PHASE3인지 알 수 없음 |
| `NULL = NULL` | 알 수 없음 | 버림 | 모르는 값 두 개가 같은지도 알 수 없음 |
| `NULL IS NULL` | 참 | 남김 | “비어 있나?”만 묻는 전용 검사 |

### 실수 2: 작은따옴표 빠뜨림

실행하면 오류 메시지가 나옵니다.

`Referenced column "COMPLETED" not found`

In [ ]:
%%sql
-- [18번] [오류 예시] 이렇게 쓰면 → Binder Error
SELECT COUNT(*) AS 완료시험
FROM studies
WHERE overall_status = COMPLETED;

In [ ]:
%%sql
-- [19번] 이렇게 고칩니다 → 'COMPLETED'
SELECT COUNT(*) AS 완료시험
FROM studies
WHERE overall_status = 'COMPLETED';

문자 값은 '작은따옴표'를 사용하고, 큰따옴표는 열 이름을 뜻합니다.

오류 메시지의 `Candidate bindings` 목록이 비슷한 열 이름을 알려 줍니다.

### 실수 3: 절의 순서를 바꿈

실행하면 오류 메시지가 나옵니다.

`Parser Error: syntax error at or near "ORDER"`

순서가 정해져 있습니다.

In [ ]:
%%sql
-- [20번] [오류 예시] 이렇게 쓰면 → Parser Error
SELECT nct_id
FROM studies
LIMIT 5
ORDER BY start_date

In [ ]:
%%sql
-- [21번] 이렇게 고칩니다 → ORDER BY 다음에 LIMIT
SELECT nct_id
FROM studies
ORDER BY start_date
LIMIT 5

SELECT → FROM → WHERE → ORDER BY → LIMIT 순서를 지킵니다.

절의 순서는 외우는 것이 아니라 “무엇을 · 어디서 · 어떤 행만 · 어떤 순서로 · 몇 줄”로 읽으면 됩니다.

### 실수 4: 대소문자

오류가 나지 않아 더 위험합니다.

`LIKE '%covid%'` 는 6건만 나옵니다. 대소문자를 가리지 않고 세면 2,125건입니다.

In [ ]:
%%sql
-- [22번] [오류 예시] 이렇게 쓰면 → 6건만
SELECT COUNT(*) AS 시험수
FROM studies
WHERE brief_title LIKE '%covid%';

In [ ]:
%%sql
-- [23번] 이렇게 고칩니다 → ILIKE
SELECT COUNT(*) AS 시험수
FROM studies
WHERE brief_title ILIKE '%covid%';

`ILIKE` 는 대소문자를 구분하지 않습니다. `UPPER()` 를 써도 됩니다.

### 실수 5: AND와 OR 섞기

오류가 나지 않아 더 위험합니다.

PHASE3은 완료 여부와 상관없이 모두 나옵니다. AND가 OR보다 먼저 계산되기 때문입니다.

In [ ]:
%%sql
-- [24번] [오류 예시] 이렇게 쓰면 → 2,798
SELECT COUNT(*) AS 시험수
FROM studies
WHERE phase = 'PHASE3'
   OR phase = 'PHASE4'
  AND overall_status = 'COMPLETED';

In [ ]:
%%sql
-- [25번] 이렇게 고칩니다 → 괄호로 묶기 → 1,818
SELECT COUNT(*) AS 시험수
FROM studies
WHERE (phase = 'PHASE3'
    OR phase = 'PHASE4')
  AND overall_status = 'COMPLETED';

괄호 안의 OR를 먼저 계산한 뒤 AND를 겁니다. `IN ('PHASE3','PHASE4')`로 써도 됩니다.

AND와 OR를 같이 쓸 때는 괄호로 묶으세요. 곱셈이 덧셈보다 먼저인 것과 같습니다.

### 실수 6: 세미콜론(;) 빠뜨림

여러 문장을 한 번에 실행할 때 생깁니다. 실행하면 오류 메시지가 나옵니다.

`Parser Error: syntax error at or near "SELECT"`

두 문장이 한 문장으로 이어져 읽힙니다.

In [ ]:
%%sql
-- [26번] [오류 예시] 이렇게 쓰면 → Parser Error
SELECT nct_id FROM studies LIMIT 3
-- ← 세미콜론 없음
SELECT nct_id, name FROM sponsors LIMIT 3;

In [ ]:
%%sql
-- [27번] 이렇게 고칩니다 → 문장마다 ;
SELECT nct_id FROM studies LIMIT 3;
-- ← 문장이 여기서 끝남
SELECT nct_id, name FROM sponsors LIMIT 3;

문장마다 끝에 `;` 를 붙여 구분합니다.

`--` 뒤의 글은 주석이라 실행되지 않습니다.

결과는 마지막 문장의 것만 보입니다.

## 직접 해보기 — 8문항

빈칸 `___` 을 채워 실행하면 결과 아래에 ✅ / ❌ 가 나옵니다.

**Q1.** SELECT · AS — “임상시험의 등록번호(`nct_id`)와 진행 상태(`overall_status`)를 `등록번호`, `진행상태` 라는 이름으로 보여 줘”

37,509행이 모두 나오면 됩니다.

In [ ]:
%%sql
-- [28번] Q1
SELECT ___ AS 등록번호,
       ___ AS 진행상태
FROM studies;

**Q2.** AND · 비교 — “등록인원(`enrollment`)이 1000명 이상이면서 완료(`COMPLETED`)된 시험은 몇 건이야?”

In [ ]:
%%sql
-- [29번] Q2
SELECT COUNT(*) AS 시험수
FROM studies
WHERE ___
  AND ___;

**Q3.** BETWEEN — “2020년에 시작한(`start_date`) 시험은 몇 건이야?”

In [ ]:
%%sql
-- [30번] Q3
SELECT COUNT(*) AS 시험수
FROM studies
WHERE start_date BETWEEN ___ AND ___;

**Q4.** IN — “2상(`PHASE2`) 또는 3상(`PHASE3`)이면서 모집 중(`RECRUITING`)인 시험은 몇 건이야?”

In [ ]:
%%sql
-- [31번] Q4
SELECT COUNT(*) AS 시험수
FROM studies
WHERE phase IN (___, ___)
  AND overall_status = ___;

**Q5.** LIKE · ILIKE — “제목(`brief_title`)에 sleep(수면)이 들어간 시험은 몇 건이야? Sleep, SLEEP 처럼 대소문자가 달라도 모두 세어 줘”

In [ ]:
%%sql
-- [32번] Q5
SELECT COUNT(*) AS 시험수
FROM studies
WHERE brief_title ___ '%sleep%';

**Q6.** IS NULL — “임상 단계(`phase`)가 비어 있는(NULL) 시험은 몇 건이야?”

In [ ]:
%%sql
-- [33번] Q6
SELECT COUNT(*) AS 단계없음
FROM studies
WHERE phase ___;

**Q7.** ORDER BY · LIMIT — “3상(`PHASE3`) 시험 중 등록인원(`enrollment`)이 가장 많은 5건을 많은 순으로 보여 줘”

In [ ]:
%%sql
-- [34번] Q7
SELECT nct_id, enrollment
FROM studies
WHERE phase = 'PHASE3'
ORDER BY ___
LIMIT ___;

**Q8.** DISTINCT — “시험 단계(`phase`)에는 어떤 값들이 있어?” 값의 종류만 중복 없이 보세요.

In [ ]:
%%sql
-- [35번] Q8
SELECT ___ phase
FROM studies;

## 6. 정리 — 용어와 요약

**7주차 용어 정리** — 이번 시간에 새로 나온 말

| 용어 | 예시 | 뜻 (한 줄) |
|---|---|---|
| `SELECT` | `SELECT nct_id, phase` | 어떤 열을 볼지 정하는 절 |
| `FROM` | `FROM studies` | 어느 테이블에서 가져올지 정하는 절 |
| `WHERE` | `WHERE phase = 'PHASE3'` | 조건에 맞는 행만 남기는 절 |
| `ORDER BY` | `ORDER BY enrollment DESC` | 결과를 정렬하는 절. DESC는 내림차순 |
| `LIMIT` | `LIMIT 10` | 앞에서 몇 줄만 보는 절 |
| `DISTINCT` | `SELECT DISTINCT phase` | 중복을 없애고 값의 종류만 보기 |
| `AS` | `nct_id AS 등록번호` | 결과 열에 다른 이름을 붙이기 |
| `LIKE` · `ILIKE` | `LIKE '%Nurs%'` | 글자 패턴 찾기. ILIKE는 대소문자 무시 |
| `IS NULL` | `WHERE 날짜 IS NULL` | 값이 비어 있는 행 찾기 |

## 오늘 배운 것

- SQL 3가지, DDL로 표를 만들고, DML로 꺼내고, DCL로 권한을 정한다
- SELECT → FROM → WHERE → ORDER BY → LIMIT 순서로 적는다(처리는 FROM → WHERE → SELECT → ORDER BY → LIMIT 순)
- 조건을 하나씩 더하며 건수가 어떻게 변하는지 확인한다
- 오류는 원인 한 줄을 읽고, 0행이면 조건부터 의심한다

다음 시간: 8강. SQL (2) — 집계(COUNT · AVG)와 GROUP BY, 그리고 JOIN

아래 셀로 8문항 결과를 확인합니다.

In [ ]:
진행확인()   # [36번]